# LeetCode 722 — Remove Comments bằng Python

Link đề: https://leetcode.com/problems/remove-comments/ · Độ khó: **Medium** · Chủ đề: Array, String

Skeleton LeetCode:

```python
class Solution:
    def removeComments(self, source: List[str]) -> List[str]:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so giữa các cách giải, với các ca tính tay và với một **bộ sinh chương trình có đáp án biết trước**), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Các ví dụ ngoài hai ví dụ chính của đề là ví dụ **tự dựng** (đối chiếu lại đề gốc nếu cần).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mã nguồn C++ chia thành các dòng, hãy **xóa comment** `// ...` (đến hết dòng) và `/* ... */` (có thể trải qua nhiều dòng), rồi trả về mã nguồn còn lại **theo cùng định dạng** (mỗi phần tử là một dòng, không có dòng rỗng).
2. Nắm các công cụ Python: duyệt chuỗi bằng chỉ số `i` và cắt lát hai ký tự `s[i:i+2]`, `"\n".join` / `str.split`, `str.find` với vị trí bắt đầu, danh sách ký tự rồi `"".join`, và thư viện `re` (biểu thức chính quy, `re.sub`).
3. Đi từ **máy trạng thái trên toàn văn bản đã nối** → **quét từng dòng với cờ "đang trong block comment"** → **nhảy bằng `str.find`** → **một biểu thức chính quy**.
4. Hiểu mấu chốt: ta chỉ cần biết **trạng thái hiện tại** (code / line comment / block comment) và nhìn **hai ký tự** để quyết định; **comment nào bắt đầu trước thì thắng**; block comment **nuốt cả ký tự xuống dòng** nên có thể **nối hai dòng** thành một.
5. Tránh các lỗi kinh điển: xóa `//` và `/* */` ở hai lượt riêng, quên nối dòng khi block trải nhiều dòng, để `/*/` tự đóng chính nó, in cả dòng rỗng, và dùng `strip()` làm mất khoảng trắng hợp lệ.

## 2. Đề bài & ví dụ

Cho một chương trình C++, hãy xóa các comment. Mã nguồn là mảng chuỗi `source`, trong đó `source[i]` là dòng thứ `i` (kết quả tách văn bản gốc theo ký tự xuống dòng `'\n'`).

Trong C++ có hai loại comment:

- **Line comment** `"//"`: nó và mọi ký tự bên phải nó **trên cùng dòng** bị bỏ qua.
- **Block comment** `"/*"`: mọi ký tự cho đến lần xuất hiện **kế tiếp, không chồng lấn** của `"*/"` bị bỏ qua. (Các lần xuất hiện được tìm theo thứ tự đọc: từng dòng, từ trái sang phải.) Chú ý `"/*/"` **chưa** kết thúc block comment, vì phần kết thúc `*/` sẽ chồng lấn lên phần bắt đầu `/*`.

**Comment hiệu lực đầu tiên thắng:** nếu `"//"` xuất hiện trong block comment thì bị bỏ qua; nếu `"/*"` xuất hiện trong line comment hay trong block comment thì cũng bị bỏ qua.

Nếu một dòng **rỗng sau khi xóa comment** thì **không được xuất** dòng đó (mọi chuỗi trong kết quả đều không rỗng). Không có ký tự điều khiển, dấu nháy đơn hay nháy kép (nên không có chuỗi ký tự chứa dấu comment). Mọi block comment đang mở **đều được đóng** về sau. Cuối cùng, **ký tự xuống dòng ẩn có thể bị block comment xóa**: ví dụ 2 cho thấy hai dòng bị nối lại.

Trả về mã nguồn sau khi xóa comment, **cùng định dạng** (list các dòng).

**Ví dụ 1**

- Input: `source = ["/*Test program */", "int main()", "{ ", "  // variable declaration ", "int a, b, c;", "/* This is a test", "   multiline  ", "   comment for ", "   testing */", "a = b + c;", "}"]`
- Output: `["int main()", "{ ", "  ", "int a, b, c;", "a = b + c;", "}"]`
- Giải thích: `/* ... */` ở dòng 1 và dòng 6 đến 9 là block comment; dòng 4 có line comment (còn lại hai khoảng trắng đầu dòng, nên dòng đó **vẫn được giữ** là `"  "`).

**Ví dụ 2**

- Input: `source = ["a/*comment", "line", "more_comment*/b"]`
- Output: `["ab"]`
- Giải thích: văn bản gốc là `"a/*comment\nline\nmore_comment*/b"`. Block comment xóa luôn các ký tự xuống dòng bên trong nó nên chỉ còn `"ab"`.

**Ràng buộc:**

- `1 <= source.length <= 100`
- `0 <= source[i].length <= 80`
- `source[i]` gồm các ký tự ASCII in được.
- Mọi block comment đang mở đều được đóng.
- Không có dấu nháy đơn hay nháy kép trong input.

In [1]:
from typing import List, Optional
import itertools
import random
import re

ex1 = ["/*Test program */", "int main()", "{ ", "  // variable declaration ", "int a, b, c;",
       "/* This is a test", "   multiline  ", "   comment for ", "   testing */", "a = b + c;", "}"]
ex1_out = ["int main()", "{ ", "  ", "int a, b, c;", "a = b + c;", "}"]
ex2 = ["a/*comment", "line", "more_comment*/b"]
ex2_out = ["ab"]

examples = [
    (ex1, ex1_out),
    (ex2, ex2_out),
]
extra_examples = [  # tự dựng, đã kiểm chứng; nên đối chiếu lại đề gốc
    (["a//b", "c"], ["a", "c"]),
    (["//only"], []),
    (["a/*b*/c"], ["ac"]),
]
for src, expected in examples + extra_examples:
    print(src, "-> kỳ vọng", expected)

['/*Test program */', 'int main()', '{ ', '  // variable declaration ', 'int a, b, c;', '/* This is a test', '   multiline  ', '   comment for ', '   testing */', 'a = b + c;', '}'] -> kỳ vọng ['int main()', '{ ', '  ', 'int a, b, c;', 'a = b + c;', '}']
['a/*comment', 'line', 'more_comment*/b'] -> kỳ vọng ['ab']
['a//b', 'c'] -> kỳ vọng ['a', 'c']
['//only'] -> kỳ vọng []
['a/*b*/c'] -> kỳ vọng ['ac']


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi bằng chỉ số và nhìn trước **hai** ký tự

Dấu comment gồm **hai** ký tự (`//`, `/*`, `*/`), nên ta không thể xét từng ký tự riêng lẻ. Ta dùng chỉ số `i` và cắt lát `s[i:i+2]` để lấy cặp ký tự bắt đầu ở `i` (khi `i` là vị trí cuối, lát cắt chỉ có **một** ký tự, nên không bao giờ bằng một dấu hai ký tự; Python không báo lỗi vượt biên với phép cắt lát).

In [2]:
s = "a/*b*/c"
print(s[0:2], s[1:3], s[5:7], s[6:8])       # 'a/' '/*' '/c' 'c'   (lát cuối chỉ có 1 ký tự)
print(s[6:8] == "*/", s[4:6] == "*/")        # False True
i = 1
while i < len(s):
    print(i, repr(s[i:i + 2]))
    i += 1

a/ /* /c c
False True
1 '/*'
2 '*b'
3 'b*'
4 '*/'
5 '/c'
6 'c'


### 3.2. Máy trạng thái (state machine) và biến cờ

Bài này chỉ cần nhớ **mình đang ở đâu**: ngoài comment (`code`), trong line comment (`line`) hay trong block comment (`block`). Mỗi ký tự (hoặc cặp ký tự) quyết định ta **ở yên** hay **chuyển trạng thái**. Một biến cờ `in_block = True/False` là bản rút gọn của máy trạng thái khi ta xử lý từng dòng (line comment chỉ sống trong một dòng nên không cần nhớ).

In [3]:
# Ví dụ máy trạng thái nhỏ: đếm số ký tự "ngoài ngoặc" trong "ab(cd)e(f)"
state = "out"
count = 0
for ch in "ab(cd)e(f)":
    if state == "out":
        if ch == "(":
            state = "in"
        else:
            count += 1
    else:
        if ch == ")":
            state = "out"
print(count)         # 3  (a, b, e)

3


### 3.3. Nối và tách dòng: `"\n".join` và `str.split`

`"\n".join(lines)` nối các dòng thành một văn bản, cách nhau bằng ký tự xuống dòng; `text.split("\n")` làm điều ngược lại. Với văn bản đã nối, block comment **nuốt cả `\n`** nên các dòng tự động được nối, và ta chỉ cần tách lại ở cuối. Lưu ý `"".split("\n")` cho `[""]` (một dòng rỗng), nên ta phải **lọc bỏ dòng rỗng**.

In [4]:
source = ["a/*comment", "line", "more_comment*/b"]
text = "\n".join(source)
print(repr(text))
print(text.split("\n") == source)            # True
print("".split("\n"), "a\n\nb".split("\n"))   # [''] ['a', '', 'b']
print([x for x in "a\n\nb".split("\n") if x])  # ['a', 'b']

'a/*comment\nline\nmore_comment*/b'
True
[''] ['a', '', 'b']
['a', 'b']


### 3.4. `str.find(sub, start)`: tìm kể từ một vị trí

`text.find(sub, start)` trả về **vị trí đầu tiên** của `sub` trong `text` kể từ chỉ số `start`, hoặc `-1` nếu không có. Dùng để **nhảy** thẳng tới dấu comment kế tiếp thay vì đi từng ký tự.

In [5]:
text = "ab//cd/*e*/f"
print(text.find("//"), text.find("/*"), text.find("*/"))      # 2 6 9
print(text.find("/*", 7), text.find("*/", 8), text.find("zz"))  # -1 9 -1

2 6 9
-1 9 -1


### 3.5. Biểu thức chính quy: `re.sub`

`re.sub(mẫu, thay_thế, văn_bản)` thay **mọi** đoạn khớp mẫu. Một vài ký hiệu cần dùng: `[^\n]*` là "mọi ký tự **trừ** xuống dòng, lặp tùy ý"; `[\s\S]*?` là "mọi ký tự **kể cả** xuống dòng, lặp **tối thiểu** (lười, `?`)"; dấu `|` là "hoặc" và **thử vế trái trước**; `\*` là ký tự `*` thật. Điều quan trọng: `re.sub` quét **từ trái sang phải** và tại mỗi vị trí chọn khớp **đầu tiên** có thể, đúng tinh thần "comment bắt đầu trước thì thắng".

In [6]:
print(re.sub(r"\d+", "#", "a1b22c333"))                  # a#b#c#
print(re.sub(r"/\*[\s\S]*?\*/", "", "x/* a\nb */y"))      # xy
print(re.sub(r"/\*.*?\*/", "", "x/* a\nb */y"))           # ‘.’ không khớp xuống dòng nên không xóa được

a#b#c#
xy
x/* a
b */y


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy **đọc như một trình biên dịch**: từ trái sang phải, từng ký tự, luôn biết mình đang ở trong comment hay không.

Lấy ví dụ 2: `"a/*comment\nline\nmore_comment*/b"`.

| Vị trí đọc | Trạng thái | Hành động | Kết quả giữ lại |
|---|---|---|---|
| `a` | code | giữ | `a` |
| `/*` | code → block | bỏ cặp `/*`, vào block | `a` |
| `comment`, `\n`, `line`, `\n`, `more_comment` | block | bỏ hết (kể cả `\n`) | `a` |
| `*/` | block → code | bỏ cặp `*/`, ra khỏi block | `a` |
| `b` | code | giữ | `ab` |

Hai điều rút ra:

1. Trong **block**, ta chỉ tìm `*/` (mọi `//` và `/*` đều bị phớt lờ).
2. Trong **code**, ta xét cặp ký tự: `//` ⇒ bỏ **đến hết dòng** (nhưng **giữ** ký tự xuống dòng); `/*` ⇒ vào block; ngược lại giữ ký tự.

Còn chuyện **xuống dòng ẩn**: vì block comment nuốt cả `\n` bên trong, hai dòng "dính" nhau như `a` và `b` ở trên. Khi xử lý **từng dòng riêng lẻ**, ta phải **nhớ phần đã giữ** của các dòng trước nếu đang kẹt trong block, và chỉ "xuất dòng" khi dòng đã kết thúc **ngoài block**.

In [7]:
# Mô phỏng tay ví dụ 2 trên văn bản đã nối
text = "\n".join(ex2)
state, kept, i = "code", "", 0
while i < len(text):
    two = text[i:i + 2]
    if state == "code":
        if two == "/*":
            state, i = "block", i + 2
        else:
            kept, i = kept + text[i], i + 1
    else:
        if two == "*/":
            state, i = "code", i + 2
        else:
            i += 1
print(repr(kept))      # 'ab'

'ab'


## 5. Cách 1 — Brute Force / baseline: máy trạng thái trên toàn văn bản đã nối

**Ý tưởng.** Nối mọi dòng thành một văn bản bằng `"\n"`, rồi chạy máy ba trạng thái từng ký tự:

- `code`: nếu `text[i:i+2] == "//"` ⇒ sang `line` (bỏ 2 ký tự); nếu `== "/*"` ⇒ sang `block` (bỏ 2 ký tự); ngược lại **giữ** ký tự.
- `line`: bỏ mọi ký tự cho tới `\n`; gặp `\n` thì **giữ** `\n` và quay về `code`.
- `block`: nếu `text[i:i+2] == "*/"` ⇒ về `code` (bỏ 2 ký tự); ngược lại bỏ 1 ký tự (kể cả `\n`).

Cuối cùng tách kết quả theo `\n` và **bỏ các dòng rỗng**.

In [8]:
def remove_comments_joined(source):
    """Cách 1: máy ba trạng thái trên văn bản đã nối."""
    text = "\n".join(source)
    out = []
    i, n = 0, len(text)
    state = "code"                       # "code" | "line" | "block"
    while i < n:
        two = text[i:i + 2]
        if state == "code":
            if two == "//":
                state = "line"
                i += 2
            elif two == "/*":
                state = "block"
                i += 2
            else:
                out.append(text[i])
                i += 1
        elif state == "line":
            if text[i] == "\n":
                state = "code"
                out.append("\n")         # giữ ký tự xuống dòng
            i += 1
        else:                            # block
            if two == "*/":
                state = "code"
                i += 2
            else:
                i += 1                   # kể cả "\n" bên trong block
    return [line for line in "".join(out).split("\n") if line]


for src, expected in examples + extra_examples:
    got = remove_comments_joined(src)
    print(got == expected, got)

True ['int main()', '{ ', '  ', 'int a, b, c;', 'a = b + c;', '}']
True ['ab']
True ['a', 'c']
True []
True ['ac']


**Độ phức tạp.** Mỗi ký tự được xét **một lần** (chỉ số `i` luôn tăng), mỗi bước cắt lát `O(1)` (hai ký tự). Gọi `L` là tổng độ dài mã nguồn: Time `O(L)`, Space `O(L)` cho văn bản nối và danh sách ký tự giữ lại.

**Nút thắt.** Đúng nhưng hơi "thừa": ta phải **nối, tách lại, lọc** dòng rỗng. Cách tiếp theo xử lý **từng dòng**, giữ một **cờ** nhớ "đang kẹt trong block" và một **bộ đệm** các ký tự đã giữ, giống như cách một trình biên dịch đọc từng dòng.

## 6. Ý tưởng thuật toán / bước đột phá

Ba nhận xét:

1. **Hai thứ cần nhớ khi chuyển từ dòng này sang dòng sau:** (a) cờ `in_block` (còn kẹt trong block comment không); (b) bộ đệm `buf` chứa phần mã **đã giữ** của dòng hiện tại. Line comment không cần nhớ vì nó kết thúc cùng dòng.
2. **Quy tắc xuất dòng:** sau khi quét xong một dòng, nếu **không còn kẹt trong block** và `buf` **không rỗng** thì xuất `buf` thành một dòng và xóa `buf`. Nếu còn kẹt trong block, **không xuất**, giữ `buf` để nối với dòng sau (đây chính là hiện tượng "xuống dòng ẩn bị xóa").
3. **Nhảy thay vì bước:** với văn bản dài, ta có thể dùng `str.find` để nhảy thẳng tới dấu comment kế tiếp; hoặc để thư viện `re` làm việc với một mẫu duy nhất vì quy tắc "comment nào bắt đầu trước thì thắng" chính là cách `re` chọn khớp ở vị trí trái nhất.

## 7. Các cách tiếp theo

### Cách 2 — Quét từng dòng với cờ `in_block` và bộ đệm (cách chuẩn)

**Giải thích từng đoạn.**

- `ans` là danh sách dòng kết quả; `buf` (list ký tự) là phần giữ lại của dòng hiện tại; `in_block` là cờ.
- Với mỗi dòng `s` (độ dài `m`), duyệt `i` từ `0`:
  - Nếu `in_block`: nếu `i + 1 < m` và `s[i:i+2] == "*/"` thì `in_block = False` và `i += 1` (bỏ thêm một ký tự). Còn lại không làm gì.
  - Nếu không `in_block`: nếu `s[i:i+2] == "/*"` thì `in_block = True` và `i += 1`; nếu `s[i:i+2] == "//"` thì `break` (bỏ cả phần còn lại của dòng); ngược lại `buf.append(s[i])`.
  - Cuối mỗi vòng `i += 1`.
- Sau khi quét xong dòng: `if not in_block and buf:` thì `ans.append("".join(buf))` và `buf.clear()`.

Chú ý `i + 1 < m` để không xét cặp vượt quá cuối dòng; thao tác `i += 1` thêm ở nhánh "tiêu thụ hai ký tự" bảo đảm **không chồng lấn** (đó là lý do `/*/` không tự đóng).

In [9]:
def remove_comments_demo(source):
    ans, buf, in_block = [], [], False
    for ln, s in enumerate(source):
        i, m = 0, len(s)
        while i < m:
            if in_block:
                if i + 1 < m and s[i:i + 2] == "*/":
                    in_block = False
                    i += 1
            else:
                if i + 1 < m and s[i:i + 2] == "/*":
                    in_block = True
                    i += 1
                elif i + 1 < m and s[i:i + 2] == "//":
                    break
                else:
                    buf.append(s[i])
            i += 1
        action = "-"
        if not in_block and buf:
            ans.append("".join(buf))
            buf.clear()
            action = "xuất dòng"
        print(f"dòng {ln}: {s!r:<22} in_block={in_block!s:<5} buf={''.join(buf)!r:<8} {action}")
    return ans


print(remove_comments_demo(ex2))

dòng 0: 'a/*comment'           in_block=True  buf='a'      -
dòng 1: 'line'                 in_block=True  buf='a'      -
dòng 2: 'more_comment*/b'      in_block=False buf=''       xuất dòng
['ab']


In [10]:
class Solution2:
    def removeComments(self, source: List[str]) -> List[str]:
        ans = []
        buf = []
        in_block = False
        for s in source:
            i, m = 0, len(s)
            while i < m:
                if in_block:
                    if i + 1 < m and s[i:i + 2] == "*/":
                        in_block = False
                        i += 1
                else:
                    if i + 1 < m and s[i:i + 2] == "/*":
                        in_block = True
                        i += 1
                    elif i + 1 < m and s[i:i + 2] == "//":
                        break
                    else:
                        buf.append(s[i])
                i += 1
            if not in_block and buf:
                ans.append("".join(buf))
                buf.clear()
        return ans


for src, expected in examples + extra_examples:
    assert Solution2().removeComments(src) == expected
print("Cách 2 đúng trên các ví dụ")

Cách 2 đúng trên các ví dụ


**Độ phức tạp Cách 2:** mỗi ký tự được xét tối đa một lần ⇒ Time `O(L)`, Space `O(L)` (kết quả và bộ đệm, không cần văn bản nối).

---

### Cách 3 — Nhảy bằng `str.find` trên văn bản đã nối

**Ý tưởng.** Thay vì xét từng ký tự, ta tìm **dấu comment kế tiếp**: `a = text.find("//", i)` và `b = text.find("/*", i)`. Điểm bắt đầu comment thực sự là cái **nhỏ hơn** (không phải `-1`). Ta sao chép nguyên khúc mã `text[i:p]`, rồi **nhảy**:

- Nếu là `//`: nhảy tới ký tự `\n` kế tiếp (giữ lại `\n`); nếu không còn `\n` thì dừng.
- Nếu là `/*`: nhảy tới **sau** `*/` đầu tiên tìm được kể từ `p + 2` (để `/*/` không tự đóng).
- Nếu không còn dấu comment nào: sao chép phần còn lại rồi dừng.

Như thế các đoạn mã bình thường được sao chép bằng một lát cắt (thay vì từng ký tự), và các comment được **bỏ qua một phát**.

In [11]:
def remove_comments_find(source):
    """Cách 3: nhảy bằng str.find trên văn bản đã nối."""
    text = "\n".join(source)
    out = []
    i = 0
    while i < len(text):
        a = text.find("//", i)
        b = text.find("/*", i)
        starts = [p for p in (a, b) if p != -1]
        if not starts:
            out.append(text[i:])
            break
        p = min(starts)
        out.append(text[i:p])                    # đoạn mã bình thường trước comment
        if p == a:                               # line comment: nhảy tới "\n" (giữ "\n")
            nl = text.find("\n", p)
            if nl == -1:
                break
            i = nl
        else:                                    # block comment: nhảy qua "*/" đầu tiên sau "/*"
            end = text.find("*/", p + 2)
            if end == -1:
                break
            i = end + 2
    return [line for line in "".join(out).split("\n") if line]


for src, expected in examples + extra_examples:
    got = remove_comments_find(src)
    print(got == expected, got)

True ['int main()', '{ ', '  ', 'int a, b, c;', 'a = b + c;', '}']
True ['ab']
True ['a', 'c']
True []
True ['ac']


**Độ phức tạp Cách 3:** mỗi lần lặp tiêu thụ ít nhất một comment hoặc kết thúc; tuy mỗi vòng gọi `find` hai lần (có thể quét xa), tổng chi phí vẫn gần `O(L)` trong thực tế (và `O(L · số_comment)` tệ nhất nếu `find` của dấu không xuất hiện phải quét hết phần còn lại mỗi vòng; với `L ≤ 8000` không đáng kể). Space `O(L)`.

---

### Cách 4 — Một biểu thức chính quy

**Ý tưởng.** Quy tắc "comment bắt đầu trước thì thắng" chính là cách `re` quét: từ trái sang phải, tại mỗi vị trí thử các lựa chọn theo thứ tự. Mẫu:

```
//[^\n]*          # line comment: // rồi mọi ký tự trừ xuống dòng
|                 # hoặc
/\*[\s\S]*?\*/    # block comment: /* rồi mọi ký tự (kể cả xuống dòng), lười, tới */ đầu tiên
```

- `[\s\S]*?` lười (non-greedy) nên dừng ở `*/` **đầu tiên**.
- Sau `/\*` ta mới bắt đầu tìm `\*/`, nên `/*/` **không** tự đóng (đúng yêu cầu "không chồng lấn").
- `re.sub(mẫu, "", text)` xóa mọi comment; sau đó tách dòng và lọc rỗng. Ký tự `\n` trong block bị xóa, trong line comment được giữ (mẫu line comment không ăn `\n`).

In [12]:
COMMENT = re.compile(r"//[^\n]*|/\*[\s\S]*?\*/")


def remove_comments_regex(source):
    """Cách 4: một biểu thức chính quy."""
    text = COMMENT.sub("", "\n".join(source))
    return [line for line in text.split("\n") if line]


for src, expected in examples + extra_examples:
    got = remove_comments_regex(src)
    print(got == expected, got)
print(COMMENT.sub("", "a/*/ x */b"))       # 'ab' : "/*/" chưa đóng

True ['int main()', '{ ', '  ', 'int a, b, c;', 'a = b + c;', '}']
True ['ab']
True ['a', 'c']
True []
True ['ac']
ab


In [13]:
class Solution4:
    def removeComments(self, source: List[str]) -> List[str]:
        import re
        text = re.sub(r"//[^\n]*|/\*[\s\S]*?\*/", "", "\n".join(source))
        return [line for line in text.split("\n") if line]


for src, expected in examples + extra_examples:
    assert Solution4().removeComments(src) == expected
print("Cách 4 đúng trên các ví dụ")

Cách 4 đúng trên các ví dụ


**Độ phức tạp Cách 4:** `re` quét tuyến tính (mỗi ký tự được xét một số lần hằng); Time `O(L)`, Space `O(L)`. Ngắn nhất, nhưng **khó mở rộng** (nếu đề có thêm dấu nháy, chuỗi ký tự, phải sửa mẫu) và người mới khó đọc.

Cách 2 và Cách 1 cùng ý tưởng "máy trạng thái"; khác ở chỗ Cách 2 **xử lý theo dòng với cờ** (không nối/tách văn bản). Cách 3 là "nhảy theo dấu comment", Cách 4 là "mô tả comment bằng mẫu". Cả bốn đều đúng.

## 8. TRACE từng bước

**Bảng trạng thái cho Cách 2 với `source = ["a/*comment", "line", "more_comment*/b"]`:**

| Dòng | Nội dung | Diễn biến khi quét | `in_block` cuối dòng | `buf` cuối dòng | Xuất? |
|---|---|---|---|---|---|
| 0 | `a/*comment` | `a` giữ; `/*` vào block; phần còn lại bỏ | `True` | `a` | không (đang trong block) |
| 1 | `line` | đang trong block, không thấy `*/` | `True` | `a` | không |
| 2 | `more_comment*/b` | trong block đến `*/` thì ra block; `b` giữ | `False` | `ab` | **có**: `"ab"` |

Kết quả: `["ab"]`.

**Hàm `trace_scan` dưới đây in từng ký tự** cho một ví dụ có đủ các tình huống: block đóng ngay trong dòng, line comment cắt dòng, và block trải nhiều dòng.

In [14]:
def trace_scan(source):
    ans, buf, in_block = [], [], False
    for ln, s in enumerate(source):
        print(f"--- dòng {ln}: {s!r}")
        i, m = 0, len(s)
        while i < m:
            two = s[i:i + 2] if i + 1 < m else s[i]
            if in_block:
                if i + 1 < m and s[i:i + 2] == "*/":
                    in_block = False
                    print(f"  i={i} {two!r}: trong block, gặp '*/' -> ra block")
                    i += 1
                else:
                    print(f"  i={i} {s[i]!r}: trong block, bỏ")
            else:
                if i + 1 < m and s[i:i + 2] == "/*":
                    in_block = True
                    print(f"  i={i} {two!r}: gặp '/*' -> vào block")
                    i += 1
                elif i + 1 < m and s[i:i + 2] == "//":
                    print(f"  i={i} {two!r}: gặp '//' -> bỏ cả phần còn lại của dòng")
                    break
                else:
                    buf.append(s[i])
                    print(f"  i={i} {s[i]!r}: giữ  (buf={''.join(buf)!r})")
            i += 1
        if not in_block and buf:
            ans.append("".join(buf))
            print(f"  => xuất dòng {''.join(buf)!r}")
            buf.clear()
        else:
            print(f"  => chưa xuất (in_block={in_block}, buf={''.join(buf)!r})")
    return ans


res = trace_scan(["x/*a*/y//z", "p/*q", "r*/s", "t"])
print(res)
assert res == ["xy", "ps", "t"]

--- dòng 0: 'x/*a*/y//z'
  i=0 'x': giữ  (buf='x')
  i=1 '/*': gặp '/*' -> vào block
  i=3 'a': trong block, bỏ
  i=4 '*/': trong block, gặp '*/' -> ra block
  i=6 'y': giữ  (buf='xy')
  i=7 '//': gặp '//' -> bỏ cả phần còn lại của dòng
  => xuất dòng 'xy'
--- dòng 1: 'p/*q'
  i=0 'p': giữ  (buf='p')
  i=1 '/*': gặp '/*' -> vào block
  i=3 'q': trong block, bỏ
  => chưa xuất (in_block=True, buf='p')
--- dòng 2: 'r*/s'
  i=0 'r': trong block, bỏ
  i=1 '*/': trong block, gặp '*/' -> ra block
  i=3 's': giữ  (buf='ps')
  => xuất dòng 'ps'
--- dòng 3: 't'
  i=0 't': giữ  (buf='t')
  => xuất dòng 't'
['xy', 'ps', 't']


**Trace nhanh Cách 3** cho `text = "x/*a*/y//z\np/*q\nr*/s\nt"`: `i = 0`: `a = 8` (`//` ở vị trí 8), `b = 1` ⇒ `p = 1` là `/*`: sao chép `"x"`, `end = text.find("*/", 3) = 4` ⇒ `i = 6`. `i = 6`: `a = 8`, `b = 12` (`/*` ở dòng 2) ⇒ `p = 8` là `//`: sao chép `text[6:8] = "y"`, nhảy tới `\n` (vị trí 10), giữ `\n`. `i = 10`: tiếp `/*` ở dòng 2: sao chép `"\np"`, nhảy qua `*/` ở dòng 3 ⇒ tiếp tục `"s\nt"`. Chuỗi giữ lại là `"xy\nps\nt"`, tách thành `["xy", "ps", "t"]`.

In [15]:
assert remove_comments_find(["x/*a*/y//z", "p/*q", "r*/s", "t"]) == ["xy", "ps", "t"]
assert remove_comments_joined(["x/*a*/y//z", "p/*q", "r*/s", "t"]) == ["xy", "ps", "t"]
assert remove_comments_regex(["x/*a*/y//z", "p/*q", "r*/s", "t"]) == ["xy", "ps", "t"]
print("4 cách cùng cho ['xy', 'ps', 't']")

4 cách cùng cho ['xy', 'ps', 't']


## 9. Code nộp LeetCode

Ta nộp **Cách 2**: xử lý từng dòng với cờ và bộ đệm, dễ chứng minh, không phụ thuộc thư viện, `O(L)`. Trên LeetCode `List` có sẵn.

In [16]:
class Solution:
    def removeComments(self, source: List[str]) -> List[str]:
        ans = []
        buf = []                     # phần mã đã giữ của dòng đang xử lý
        in_block = False             # đang kẹt trong block comment?
        for s in source:
            i, m = 0, len(s)
            while i < m:
                if in_block:
                    if i + 1 < m and s[i:i + 2] == "*/":
                        in_block = False
                        i += 1       # tiêu thụ cả hai ký tự "*/"
                else:
                    if i + 1 < m and s[i:i + 2] == "/*":
                        in_block = True
                        i += 1       # tiêu thụ cả hai ký tự "/*" (nên "/*/" không tự đóng)
                    elif i + 1 < m and s[i:i + 2] == "//":
                        break        # bỏ phần còn lại của dòng
                    else:
                        buf.append(s[i])
                i += 1
            if not in_block and buf:         # dòng kết thúc ngoài block và không rỗng
                ans.append("".join(buf))
                buf.clear()
        return ans


for src, expected in examples + extra_examples:
    assert Solution().removeComments(src) == expected
print("Code nộp đúng trên 5 ví dụ")

Code nộp đúng trên 5 ví dụ


## 10. Vì sao thuật toán đúng

**Bất biến.** Sau khi xử lý xong các ký tự `0..i-1` của một dòng (và mọi dòng trước), `in_block` cho biết **đúng** việc vị trí `i` có nằm trong một block comment hay không, và `buf` chứa **đúng** các ký tự mã (ngoài comment) của "dòng ảo" hiện tại tính đến vị trí đó.

- **Khởi đầu:** `in_block = False`, `buf` rỗng: đúng.
- **Trong block:** ký tự duy nhất thay đổi trạng thái là cặp `*/`. Ta xét `s[i:i+2] == "*/"` với `i` hiện tại; vì ta chỉ vào block bằng cách tiêu thụ **cả hai** ký tự `/*` (thêm `i += 1`), cặp `*/` mà ta tìm luôn bắt đầu **sau** `/*`: tức là `/*/` không bị coi là đóng, đúng mô tả "không chồng lấn". Mọi ký tự khác bị bỏ (kể cả `//` và `/*` bên trong): đúng luật "comment hiệu lực đầu tiên thắng".
- **Ngoài block:** nếu gặp `/*` ta vào block; nếu gặp `//` ta bỏ nốt phần còn lại của dòng (line comment không vượt qua dòng) bằng `break`; còn lại ký tự là mã và được giữ. Thứ tự kiểm tra `/*` rồi `//` không quan trọng vì hai dấu bắt đầu bằng cùng ký tự `/` nhưng **ký tự thứ hai khác nhau**, nên không thể cùng khớp ở một vị trí. Cặp `*/` ngoài block **không** đặc biệt, nên được giữ như hai ký tự mã (đúng với C++).
- **Xuất dòng.** Một "dòng ảo" kết thúc khi gặp ký tự xuống dòng **nằm ngoài block** (vì nếu nó nằm trong block thì bị nuốt). Cuối mỗi dòng gốc, nếu `in_block = False` thì ký tự xuống dòng của dòng đó nằm ngoài block: dòng ảo kết thúc, xuất `buf` nếu không rỗng (yêu cầu không xuất dòng rỗng). Nếu `in_block = True` thì ký tự xuống dòng nằm trong block, bị xóa: dòng ảo **tiếp tục** sang dòng sau, và `buf` được giữ để nối.
- **Line comment không "chảy" sang dòng sau:** `break` chỉ thoát vòng của dòng hiện tại; dòng kế tiếp bắt đầu với cùng `in_block` (vẫn `False`, vì line comment chỉ được nhận ra ngoài block).

**Các cách khác.** Cách 1 mô phỏng cùng luật nhưng trên văn bản nối (ký tự `\n` được xử lý tường minh). Cách 3 nhảy tới dấu bắt đầu comment **trái nhất** (`min` của hai `find`): đó chính là "comment đầu tiên thắng"; rồi nhảy tới điểm kết thúc đúng luật. Cách 4: bộ máy `re` chọn khớp ở vị trí trái nhất, thử vế trái của `|` trước, và `/\*[\s\S]*?\*/` bắt đầu tìm `*/` sau `/*`. Ở mục 12 ta đối chiếu cả bốn cách với một **bộ sinh chương trình có đáp án biết trước theo cách xây dựng** (không dùng thuật toán nào ở trên).

## 11. Phân tích Time / Space Complexity

Gọi `L` là tổng độ dài mã nguồn (tối đa `100 · 80 = 8000` ký tự).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Máy trạng thái trên văn bản nối | `O(L)` | `O(L)` | cần nối rồi tách lại |
| 2. Quét từng dòng + cờ (nộp) | `O(L)` | `O(L)` | không nối văn bản |
| 3. Nhảy bằng `find` | `O(L)` thực tế | `O(L)` | sao chép theo lát cắt, tệ nhất hơi hơn do `find` lặp |
| 4. Biểu thức chính quy | `O(L)` | `O(L)` | ngắn, khó mở rộng |

Cận dưới `Ω(L)` vì phải đọc mọi ký tự. Với `L ≤ 8000` cả bốn cách chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ đề, biên và các ca tính tay

Mỗi ca dưới đây có kết quả được **tính tay** theo luật (giải thích ở cột cuối):

| `source` | Kết quả | Vì sao |
|---|---|---|
| `["//only"]` | `[]` | toàn bộ dòng là comment, không xuất dòng rỗng |
| `["a/*b*/c"]` | `["ac"]` | block trong cùng dòng, hai nửa mã nối lại |
| `["a/**/b"]` | `["ab"]` | block rỗng `/**/` |
| `["a/***/b"]` | `["ab"]` | `/*` rồi `*` rồi `*/` |
| `["/*/ x */y"]` | `["y"]` | `/*/` chưa đóng; đóng ở `*/` sau |
| `["/*/", "*/", "x"]` | `["x"]` | dòng 1 `/*/` mở block (chưa đóng), dòng 2 đóng |
| `["a/*b//*c", "d*/e"]` | `["ae"]` | `//` và `/*` trong block bị bỏ qua |
| `["a//*b", "c*/d"]` | `["a", "c*/d"]` | `/*` trong line comment bị bỏ qua; `*/` ngoài block chỉ là mã |
| `["x/y*z"]` | `["x/y*z"]` | `/` và `*` đơn lẻ là mã |
| `["///"]` | `[]` | line comment |
| `["", "a", ""]` | `["a"]` | dòng rỗng không xuất |
| `[" a "]` | `[" a "]` | giữ khoảng trắng |
| `["a/*x*///y", "b"]` | `["a", "b"]` | block rồi ngay line comment |
| `["a/*", "*/", "b"]` | `["a", "b"]` | block bắt đầu ở cuối dòng và đóng ở đầu dòng sau, `a` được xuất ở dòng 2, `b` ở dòng 3 |

In [17]:
hand_cases = [
    (["//only"], []),
    (["a/*b*/c"], ["ac"]),
    (["a/**/b"], ["ab"]),
    (["a/***/b"], ["ab"]),
    (["/*/ x */y"], ["y"]),
    (["/*/", "*/", "x"], ["x"]),
    (["a/*b//*c", "d*/e"], ["ae"]),
    (["a//*b", "c*/d"], ["a", "c*/d"]),
    (["x/y*z"], ["x/y*z"]),
    (["///"], []),
    (["", "a", ""], ["a"]),
    ([" a "], [" a "]),
    (["a/*x*///y", "b"], ["a", "b"]),
    (["a/*", "*/", "b"], ["a", "b"]),
]
all_cases = examples + extra_examples + hand_cases

solvers = {
    "Cách 1 máy trạng thái": remove_comments_joined,
    "Cách 2 quét từng dòng": Solution2().removeComments,
    "Cách 3 nhảy find": remove_comments_find,
    "Cách 4 regex": remove_comments_regex,
    "Code nộp": Solution().removeComments,
}
for src, expected in all_cases:
    for name, f in solvers.items():
        assert f(list(src)) == expected, (name, src, f(list(src)), expected)
print("Tất cả", len(all_cases), "ca đều đúng với mọi cách")

Tất cả 19 ca đều đúng với mọi cách


### 12.2. Phản ví dụ cho các cách làm sai

Một cách làm "nghe có vẻ hợp lý" là **xóa mọi `//...` trước, rồi xóa mọi `/*...*/`** (hai lượt riêng). Nó sai khi `//` nằm **trong** một block comment:

In [18]:
def wrong_two_pass(source):
    lines = [s.split("//")[0] for s in source]          # lượt 1: cắt line comment ở mỗi dòng
    text = "\n".join(lines)
    while "/*" in text and "*/" in text:                 # lượt 2: cắt block comment
        a = text.index("/*")
        b = text.index("*/", a + 2)
        text = text[:a] + text[b + 2:]
    return [l for l in text.split("\n") if l]


case = ["a/* // */ b"]
print("sai :", wrong_two_pass(case))
print("đúng:", Solution().removeComments(case))
assert wrong_two_pass(case) != Solution().removeComments(case)

sai : ['a/* ']
đúng: ['a b']


Vì sao sai: ở lượt 1, `"a/* // */ b"` bị cắt thành `"a/* "` (mất phần sau `//`, kể cả `*/` và ` b`), nên block comment không còn điểm kết thúc và phần còn lại sai (đúng ra kết quả là `"a b"`). Quy tắc đúng: **`//` trong block comment bị phớt lờ**.

### 12.3. Đối chiếu bằng bộ **sinh chương trình có đáp án biết trước**

Đây là oracle **độc lập** với mọi cách giải: ta **xây** chương trình từ các mảnh và đồng thời xây **đáp án** theo cách xây dựng:

- Mảnh **mã** (`code`): từ một bảng đơn vị an toàn (chữ, khoảng trắng, `*` đơn lẻ, `x/y`) ⇒ xuất hiện nguyên trong đáp án.
- Mảnh **xuống dòng** ⇒ xuất hiện trong đáp án.
- Mảnh **line comment** `//` + thân (thân có thể chứa `/*`, `*/`, `//`) rồi **bắt buộc xuống dòng** (hoặc hết chương trình) ⇒ chỉ giữ lại ký tự xuống dòng.
- Mảnh **block comment** `/*` + thân + `*/` (thân có thể chứa `//`, `/*`, xuống dòng nhưng **không** chứa `*/`) ⇒ biến mất hoàn toàn, **kể cả xuống dòng bên trong**.

Văn bản chương trình = nối các mảnh; `source` = tách theo `\n`; đáp án = tách "văn bản mã" theo `\n` rồi bỏ dòng rỗng. Các đơn vị mã được chọn để **không** vô tình tạo dấu comment ở chỗ nối (không có đơn vị nào kết thúc bằng `/` hay bắt đầu bằng `/`).

In [19]:
CODE_UNITS = ["a", "b", "c", " ", "  ", ";", "=", "+", "*", "x/y", "a*b", "int x", "{", "}", "1"]
BODY_CHARS = ["a", "b", " ", "\n", "/", "*", "x", "//", "/*", "\n"]


def gen_program(rng, max_pieces=12):
    """Trả (source, expected) với expected biết trước theo cách xây dựng."""
    text, kept = [], []
    need_newline = False                   # sau line comment bắt buộc có xuống dòng (hoặc hết)
    for _ in range(rng.randint(1, max_pieces)):
        if need_newline:
            text.append("\n")
            kept.append("\n")
            need_newline = False
        kind = rng.choices(["code", "nl", "line", "block"], weights=[45, 20, 15, 20])[0]
        if kind == "code":
            u = rng.choice(CODE_UNITS)
            text.append(u)
            kept.append(u)
        elif kind == "nl":
            text.append("\n")
            kept.append("\n")
        elif kind == "line":
            body = "".join(rng.choice([c for c in BODY_CHARS if c != "\n"]) for _ in range(rng.randint(0, 6)))
            text.append("//" + body)
            need_newline = True
        else:
            body = "".join(rng.choice(BODY_CHARS) for _ in range(rng.randint(0, 8)))
            while "*/" in body:
                body = body.replace("*/", "* /")
            text.append("/*" + body + "*/")
    program = "".join(text)
    expected = [l for l in "".join(kept).split("\n") if l]
    return program.split("\n"), expected


rng = random.Random(722)
for _ in range(3):
    src, exp = gen_program(rng)
    print(src, "->", exp)

['//*', '/*/ /', '', '/**/1/*//bba*/;/*', '', '', 'ab', '*////xab', '/*a', 'x*a', 'x*/a*b/*', 'bxb///*/'] -> ['1;', 'a*b']
['    /*/*ax', '/', ' */'] -> ['    ']
['1a'] -> ['1a']


In [20]:
rng = random.Random(7220)
n_cases = 0
n_multiline_block = 0
for _ in range(5000):
    src, expected = gen_program(rng)
    assert all(len(s) <= 10 ** 4 for s in src)
    n_multiline_block += any("/*" in s and "*/" not in s for s in src)
    for name, f in solvers.items():
        got = f(list(src))
        assert got == expected, (name, src, got, expected)
    n_cases += 1
print("Đã đối chiếu", n_cases, "chương trình sinh ngẫu nhiên (seed 7220) với đáp án xây dựng sẵn")
print("trong đó có", n_multiline_block, "chương trình có block comment trải nhiều dòng")
assert n_multiline_block > 300

Đã đối chiếu 5000 chương trình sinh ngẫu nhiên (seed 7220) với đáp án xây dựng sẵn
trong đó có 2198 chương trình có block comment trải nhiều dòng


### 12.4. Cỡ lớn (đúng ràng buộc) và chương trình dài

`100` dòng, mỗi dòng `80` ký tự; thử các dạng cực đoan: toàn mã, toàn line comment, một block comment dài phủ hết, và nhiều block xen kẽ.

In [21]:
big = {
    "toàn mã": ["a" * 80 for _ in range(100)],
    "toàn line comment": ["//" + "x" * 78 for _ in range(100)],
    "một block phủ hết": ["/*" + "y" * 78] + ["z" * 80 for _ in range(98)] + ["w" * 78 + "*/"],
    "block xen kẽ": [("/*" + "q" * 30 + "*/" + "k" * 46) for _ in range(100)],
    "toàn dòng rỗng": ["" for _ in range(100)],
}
for name, src in big.items():
    assert all(len(s) <= 80 for s in src) and len(src) <= 100
    ref = Solution().removeComments(list(src))
    for sname, f in solvers.items():
        assert f(list(src)) == ref, (name, sname)
    print(f"{name:<20} -> {len(ref)} dòng kết quả")
assert Solution().removeComments(big["toàn mã"]) == big["toàn mã"]
assert Solution().removeComments(big["toàn line comment"]) == []
assert Solution().removeComments(big["một block phủ hết"]) == []
assert Solution().removeComments(big["block xen kẽ"]) == ["k" * 46] * 100
assert Solution().removeComments(big["toàn dòng rỗng"]) == []

toàn mã              -> 100 dòng kết quả
toàn line comment    -> 0 dòng kết quả
một block phủ hết    -> 0 dòng kết quả
block xen kẽ         -> 100 dòng kết quả
toàn dòng rỗng       -> 0 dòng kết quả


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Xóa `//` và `/* */` ở hai lượt riêng

Như đã thấy ở 12.2: lượt cắt line comment trước làm hỏng block comment chứa `//`. **Sửa:** quét **một lượt** từ trái sang phải, xử lý "comment nào bắt đầu trước thì thắng" bằng trạng thái.

### Lỗi 2: Xuất dòng ngay cả khi đang kẹt trong block (quên nối dòng)

Xử lý mỗi dòng độc lập rồi thêm vào kết quả sẽ **không nối** `a` và `b` trong ví dụ 2.

In [22]:
def sai_khong_noi_dong(source):
    ans, in_block = [], False
    for s in source:
        buf, i, m = [], 0, len(s)           # SAI: buf bị tạo lại mỗi dòng
        while i < m:
            if in_block:
                if i + 1 < m and s[i:i + 2] == "*/":
                    in_block = False
                    i += 1
            else:
                if i + 1 < m and s[i:i + 2] == "/*":
                    in_block = True
                    i += 1
                elif i + 1 < m and s[i:i + 2] == "//":
                    break
                else:
                    buf.append(s[i])
            i += 1
        if buf:
            ans.append("".join(buf))        # SAI: xuất cả khi còn trong block
    return ans


print("sai :", sai_khong_noi_dong(ex2))            # ['a', 'b']
print("đúng:", Solution().removeComments(ex2))     # ['ab']
assert sai_khong_noi_dong(ex2) == ["a", "b"]

sai : ['a', 'b']
đúng: ['ab']


**Vì sao sai:** block comment đã xóa ký tự xuống dòng giữa `a` và `b`; ta phải **giữ `buf` qua các dòng** và chỉ xuất khi `not in_block`. **Sửa:** tạo `buf` **một lần** ngoài vòng lặp dòng và xuất theo điều kiện `not in_block and buf`.

### Lỗi 3: Không tiêu thụ **cả hai** ký tự của `/*` (để `/*/` tự đóng)

Nếu sau khi gặp `/*` ta chỉ tăng `i` một bước, ký tự `*` của `/*` có thể được dùng lại làm phần đầu của `*/`.

In [23]:
def sai_chong_lan(source):
    ans, buf, in_block = [], [], False
    for s in source:
        i, m = 0, len(s)
        while i < m:
            if in_block:
                if i + 1 < m and s[i:i + 2] == "*/":
                    in_block = False
                    i += 1
            else:
                if i + 1 < m and s[i:i + 2] == "/*":
                    in_block = True            # SAI: quên i += 1
                elif i + 1 < m and s[i:i + 2] == "//":
                    break
                else:
                    buf.append(s[i])
            i += 1
        if not in_block and buf:
            ans.append("".join(buf))
            buf.clear()
    return ans


case = ["/*/ x", "*/y"]
print("sai :", sai_chong_lan(case))                  # block đóng ngay ở '*/' chồng lấn => lộ ' x'
print("đúng:", Solution().removeComments(case))
assert sai_chong_lan(case) != Solution().removeComments(case)

sai : [' x', '*/y']
đúng: ['y']


**Vì sao sai:** `"/*/"` bị đọc là `/*` rồi `*/` (dùng chung ký tự `*`), nên block đóng sớm và phần ` x` bị coi là mã. **Sửa:** sau khi nhận `/*` (hoặc `*/`), `i += 1` thêm một lần để **bỏ cả hai ký tự**.

### Lỗi 4: Xuất cả dòng rỗng

Đề yêu cầu "mọi chuỗi trong kết quả đều **không rỗng**". Nếu `ans.append(...)` vô điều kiện, dòng toàn comment sẽ thành `""`.

In [24]:
def sai_dong_rong(source):
    ans, buf, in_block = [], [], False
    for s in source:
        i, m = 0, len(s)
        while i < m:
            if in_block:
                if i + 1 < m and s[i:i + 2] == "*/":
                    in_block = False
                    i += 1
            else:
                if i + 1 < m and s[i:i + 2] == "/*":
                    in_block = True
                    i += 1
                elif i + 1 < m and s[i:i + 2] == "//":
                    break
                else:
                    buf.append(s[i])
            i += 1
        if not in_block:                        # SAI: thiếu `and buf`
            ans.append("".join(buf))
            buf.clear()
    return ans


print("sai :", sai_dong_rong(["//x", "a", "/* c */"]))        # ['', 'a', '']
print("đúng:", Solution().removeComments(["//x", "a", "/* c */"]))
assert sai_dong_rong(["//x", "a", "/* c */"]) == ["", "a", ""]

sai : ['', 'a', '']
đúng: ['a']


**Sửa:** điều kiện xuất là `not in_block and buf` (hoặc `if buf:` sau khi đã chắc không còn trong block).

### Lỗi 5: Dùng `strip()` làm mất khoảng trắng hợp lệ

Ví dụ 1 cho thấy dòng `"  // variable declaration "` thành `"  "` (hai khoảng trắng đầu dòng) và vẫn được **giữ**; dòng `"{ "` giữ khoảng trắng cuối. Nếu `strip()` rồi bỏ dòng rỗng, ta sẽ mất dòng `"  "` và đổi `"{ "` thành `"{"`.

In [25]:
sai = [x.strip() for x in Solution().removeComments(ex1) if x.strip()]
print("sai :", sai)
print("đúng:", Solution().removeComments(ex1))
assert sai != Solution().removeComments(ex1)

sai : ['int main()', '{', 'int a, b, c;', 'a = b + c;', '}']
đúng: ['int main()', '{ ', '  ', 'int a, b, c;', 'a = b + c;', '}']


**Sửa:** đừng chỉnh sửa khoảng trắng; chỉ **bỏ các dòng có độ dài `0`**.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "bỏ qua cho tới khi gặp dấu kết thúc", "comment/chuỗi lồng nhau", "ưu tiên cái bắt đầu trước" ⇒ **máy trạng thái** hoặc cờ boolean.
- **Dấu gồm hai ký tự:** dùng `s[i:i+2]` và **tiêu thụ cả hai** (`i += 1` thêm) để không chồng lấn.
- **Trạng thái xuyên dòng:** cờ `in_block` và bộ đệm `buf` phải sống **ngoài** vòng lặp từng dòng.
- **Xuất dòng khi đủ điều kiện:** `not in_block and buf`.
- **Nối rồi tách:** đôi khi dễ hơn là `"\n".join` → xử lý một văn bản → `split("\n")`, nhưng nhớ lọc dòng rỗng.
- **Thử các ca hiểm:** `/*/`, `/**/`, `//` trong block, `/*` trong line comment, `*/` ngoài block, block mở cuối dòng và đóng đầu dòng sau, dòng toàn khoảng trắng.
- **Kiểm chứng bằng bộ sinh có đáp án xây dựng sẵn** là kỹ thuật rất mạnh cho bài xử lý văn bản.
- Bài liên quan: **Basic Calculator** (trạng thái/dấu ngoặc), **Decode String**, **Valid Parentheses**, **Simplify Path**.

## 15. Biến thể

1. **Có chuỗi ký tự** `"..."`: dấu comment **bên trong chuỗi không có hiệu lực**; thêm trạng thái `string`.
2. **Block comment lồng nhau** (như Rust, Haskell): dùng **bộ đếm độ sâu** thay cho cờ boolean.
3. **Giữ lại số dòng gốc** (ví dụ để báo lỗi): thay vì nối các dòng, vẫn xuất dòng rỗng.
4. **Chỉ xóa line comment / chỉ xóa block comment.**
5. **Đếm số ký tự comment bị xóa** hoặc trả về danh sách vị trí `(dòng bắt đầu, dòng kết thúc)` của các block comment.

In [26]:
def remove_comments_with_strings(source):
    """Biến thể 1: chuỗi "..." (không có escape) bảo vệ dấu comment bên trong."""
    ans, buf = [], []
    state = "code"                # "code" | "block" | "string"
    for s in source:
        i, m = 0, len(s)
        while i < m:
            two = s[i:i + 2]
            if state == "block":
                if two == "*/":
                    state = "code"
                    i += 1
            elif state == "string":
                buf.append(s[i])
                if s[i] == '"':
                    state = "code"
            else:
                if two == "/*":
                    state = "block"
                    i += 1
                elif two == "//":
                    break
                else:
                    buf.append(s[i])
                    if s[i] == '"':
                        state = "string"
            i += 1
        if state == "string":      # chuỗi không xuyên dòng: kết thúc dòng thì thoát chuỗi
            state = "code"
        if state != "block" and buf:
            ans.append("".join(buf))
            buf.clear()
    return ans


print(remove_comments_with_strings(['s = "// not a comment"; // real', 'x = "/*"; /* c */ y']))
# ['s = "// not a comment"; ', 'x = "/*";  y']
assert remove_comments_with_strings(['s = "// not a comment"; // real']) == ['s = "// not a comment"; ']
assert remove_comments_with_strings(['x = "/*"; /* c */ y']) == ['x = "/*";  y']

['s = "// not a comment"; ', 'x = "/*";  y']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Máy trạng thái trên văn bản nối | Ba trạng thái, đi từng ký tự, giữ `\n` của line comment | `O(L)` | `O(L)` | Dễ hình dung nhất, dễ mở rộng thêm trạng thái |
| 2. Quét từng dòng + cờ `in_block` | Cờ + bộ đệm xuyên dòng, xuất khi `not in_block and buf` | `O(L)` | `O(L)` | **Nộp bài**: gọn, không cần nối/tách |
| 3. Nhảy bằng `str.find` | Tìm dấu comment trái nhất rồi nhảy qua | `O(L)` thực tế | `O(L)` | Văn bản rất dài, ít comment |
| 4. Biểu thức chính quy | Một mẫu `//[^\n]*|/\*[\s\S]*?\*/` | `O(L)` | `O(L)` | Viết nhanh khi đề cố định; khó mở rộng (chuỗi, lồng nhau) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().removeComments(["a//b"])`; `Solution().removeComments(["/*x*/"])`; `Solution().removeComments(["a/*b", "c*/d"])`; `"ab".split("\n")`; và `"x/*y*/z"[1:3]`.

**Bài 2 (tính tay).** Với `source = ["x/*y", "z*/w//q", "r"]`, lập bảng như mục 8 (mỗi dòng: `in_block` cuối dòng, `buf` cuối dòng, có xuất không) và cho biết kết quả.

**Bài 3 (điền TODO).** Hoàn thành `remove_comments_todo(source)` theo Cách 2.

In [27]:
def remove_comments_todo(source):
    # TODO 1: ans = []; buf = []; in_block = False
    # TODO 2: với mỗi dòng s: i = 0; while i < len(s): xử lý in_block / "/*" / "//" / ký tự mã (nhớ i += 1 thêm khi tiêu thụ 2 ký tự)
    # TODO 3: sau mỗi dòng: nếu not in_block và buf thì ans.append("".join(buf)); buf.clear()
    # TODO 4: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `remove_comments_re(source)` bằng **một** lời gọi `re.sub` (Cách 4) rồi tách và lọc dòng rỗng.

**Bài 5 (biến thể / mini challenge).** (a) Viết `block_spans(source)` trả về danh sách `(dòng bắt đầu, dòng kết thúc)` (chỉ số `0-based`) của mọi block comment hiệu lực (tức không nằm trong line comment hay block khác). (b) Viết `count_removed_chars(source)` đếm số ký tự **bị xóa** (kể cả ký tự của dấu comment; không tính ký tự xuống dòng giữa các dòng). Kiểm chứng: số ký tự bị xóa phải bằng `sum(map(len, source)) - sum(map(len, kết_quả))`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `["a"]`; `[]`; `["ad"]`; `["ab"]`; `"/*"` (cắt `[1:3]` của `"x/*y*/z"` là ký tự ở chỉ số 1 và 2). Bài 3: bám sát mục 9.
- **Mức 2 (vừa):** Bài 2: dòng 0 `"x/*y"`: `x` giữ, `/*` vào block ⇒ `in_block = True`, `buf = "x"`, không xuất. Dòng 1 `"z*/w//q"`: trong block đến `*/` ra block, `w` giữ ⇒ `buf = "xw"`, gặp `//` thì bỏ phần còn lại; cuối dòng `in_block = False` ⇒ xuất `"xw"`. Dòng 2 `"r"`: giữ, xuất `"r"`. Kết quả `["xw", "r"]`. Bài 4: mẫu `r"//[^\n]*|/\*[\s\S]*?\*/"` và đừng quên `"\n".join`.
- **Mức 3 (sâu):** Bài 5(a): duyệt từng dòng với `in_block` và `start_line`; khi vào block ngoài mọi comment (ở dòng `i`) ghi `start_line = i`; khi ra block ở dòng `j` thêm `(start_line, j)`. Một dòng có thể chứa nhiều block. Bài 5(b): trong máy trạng thái, mỗi ký tự không được giữ (kể cả ký tự của dấu comment và phần bị `break` bỏ) thì cộng `1`; mọi ký tự gốc hoặc được giữ hoặc bị xóa nên tổng số xóa đúng bằng `sum(map(len, source)) - sum(map(len, kết_quả))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert Solution().removeComments(["a//b"]) == ["a"]
assert Solution().removeComments(["/*x*/"]) == []
assert Solution().removeComments(["a/*b", "c*/d"]) == ["ad"]
assert "ab".split("\n") == ["ab"]
assert "x/*y*/z"[1:3] == "/*"

# Bài 2
assert Solution().removeComments(["x/*y", "z*/w//q", "r"]) == ["xw", "r"]


# Bài 3
def remove_comments_todo_ans(source):
    ans, buf, in_block = [], [], False
    for s in source:
        i = 0
        while i < len(s):
            if in_block:
                if s[i:i + 2] == "*/":
                    in_block = False
                    i += 1
            elif s[i:i + 2] == "/*":
                in_block = True
                i += 1
            elif s[i:i + 2] == "//":
                break
            else:
                buf.append(s[i])
            i += 1
        if not in_block and buf:
            ans.append("".join(buf))
            buf.clear()
    return ans


# Bài 4
def remove_comments_re(source):
    text = re.sub(r"//[^\n]*|/\*[\s\S]*?\*/", "", "\n".join(source))
    return [x for x in text.split("\n") if x]


# Bài 5
def block_spans(source):
    spans, in_block, start = [], False, -1
    for ln, s in enumerate(source):
        i, m = 0, len(s)
        while i < m:
            if in_block:
                if s[i:i + 2] == "*/":
                    in_block = False
                    spans.append((start, ln))
                    i += 1
            elif s[i:i + 2] == "/*":
                in_block = True
                start = ln
                i += 1
            elif s[i:i + 2] == "//":
                break
            i += 1
    return spans


def count_removed_chars(source):
    removed = 0
    in_block = False
    for s in source:
        i, m = 0, len(s)
        while i < m:
            if in_block:
                if s[i:i + 2] == "*/":
                    in_block = False
                    removed += 2
                    i += 1
                else:
                    removed += 1
            elif s[i:i + 2] == "/*":
                in_block = True
                removed += 2
                i += 1
            elif s[i:i + 2] == "//":
                removed += m - i
                break
            i += 1
    return removed


for src, expected in all_cases:
    assert remove_comments_todo_ans(list(src)) == expected
    assert remove_comments_re(list(src)) == expected
assert block_spans(ex1) == [(0, 0), (5, 8)]
assert block_spans(ex2) == [(0, 2)]
assert block_spans(["a//b/*c", "d"]) == []
assert count_removed_chars(["a//bc"]) == 4
assert count_removed_chars(ex2) == sum(map(len, ex2)) - 2
rng5 = random.Random(7221)
for _ in range(500):
    src, expected = gen_program(rng5)
    assert remove_comments_todo_ans(list(src)) == expected == remove_comments_re(list(src))
    assert count_removed_chars(src) == sum(map(len, src)) - sum(map(len, expected))
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (quét từng dòng với cờ `in_block` và bộ đệm `buf`)

```python
class Solution:
    def removeComments(self, source: List[str]) -> List[str]:
        ans, buf, in_block = [], [], False
        for s in source:
            i, m = 0, len(s)
            while i < m:
                if in_block:
                    if i + 1 < m and s[i:i + 2] == "*/":
                        in_block = False
                        i += 1                 # tiêu thụ cả hai ký tự
                else:
                    if i + 1 < m and s[i:i + 2] == "/*":
                        in_block = True
                        i += 1                 # tiêu thụ cả hai ký tự
                    elif i + 1 < m and s[i:i + 2] == "//":
                        break                  # bỏ phần còn lại của dòng
                    else:
                        buf.append(s[i])
                i += 1
            if not in_block and buf:           # xuất dòng khi ngoài block và không rỗng
                ans.append("".join(buf))
                buf.clear()
        return ans
```

**Checklist**

- [ ] `in_block` và `buf` được tạo **ngoài** vòng lặp từng dòng (để nối các dòng bị block comment gộp)?
- [ ] Gặp `/*` hoặc `*/` thì **tiêu thụ cả hai ký tự** (để `/*/` không tự đóng)?
- [ ] `//` trong block bị bỏ qua; `/*` trong line comment bị bỏ qua (line comment `break` ngay)?
- [ ] `*/` **ngoài** block là mã bình thường (không làm gì đặc biệt)?
- [ ] Chỉ xuất dòng khi `not in_block and buf` (không xuất dòng rỗng)?
- [ ] Không `strip()` hay sửa khoảng trắng (ví dụ `"  "` hay `"{ "` được giữ nguyên)?
- [ ] Đã thử ví dụ 1, ví dụ 2, `"/*/ x */y"`, `["a//*b", "c*/d"]`, `["//only"]`?
- [ ] Skeleton đúng `class Solution` / `def removeComments(self, source: List[str]) -> List[str]`?